# Prophet — Previsão do IPCA com variáveis externas defasadas

**Projeto:** IPCA sob pressão externa  
**Autor:** Gabriel Abreu  
**Orientadora:** Bruna Nogueira Rezende  

Este notebook cobre as etapas do pipeline analítico para o modelo Prophet:
1. Justificativa teórica e adequação ao IPCA
2. Preparação dos dados no formato Prophet
3. Otimização de hiperparâmetros (grid search + CV temporal nativa)
4. Seleção do modo de sazonalidade (aditivo vs. multiplicativo)
5. Expanding window — avaliação fora da amostra (h = 1, 2, 3)
6. Métricas MAE, RMSE, MASE e análise por regime de volatilidade
7. Teste de Diebold-Mariano (ganho incremental do CPI — H3)

### Por que Prophet para previsão macroeconômica?

O Prophet (Taylor & Letham, 2017) é um modelo de decomposição aditiva/multiplicativa estruturado como um problema de regressão bayesiana:

$$y_t = g(t) + s(t) + h(t) + \varepsilon_t$$

onde $g(t)$ é a tendência (linear ou logística), $s(t)$ a sazonalidade por séries de Fourier e $h(t)$ o efeito de eventos especiais. Sua vantagem é a robustez a quebras estruturais via *changepoints* automáticos, comportamento relevante para o IPCA, que apresentou rupturas documentadas em 2015-16 e 2020-22. A inclusão de regressores externos via `add_regressor` permite incorporar câmbio e CPI defasados sem alterar a estrutura nativa.

**Limitação relevante (feedback da orientadora):** o Prophet não é arquitetado como modelo autorregressivo. Os lags do IPCA são incluídos explicitamente como regressores adicionais para garantir comparabilidade com ARIMAX e LightGBM.

## 0. Dependências

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats
import itertools
import pickle

np.random.seed(42)
plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Dependências carregadas.')

## 1. Carregamento e preparação no formato Prophet

O Prophet exige colunas `ds` (datetime) e `y` (variável alvo). Regressores extras são adicionados como colunas adicionais.

In [ ]:
df_raw = pd.read_csv('final_df_with_lags.csv', parse_dates=['data'], index_col='data')
df_raw = df_raw.asfreq('MS')

print(f'Shape: {df_raw.shape}')
print(f'Período: {df_raw.index[0].date()} → {df_raw.index[-1].date()}')
print('Colunas:', df_raw.columns.tolist())

In [ ]:
def prepare_prophet_df(df, target='ipca_pct', regressor_cols=None):
    """
    Converte DataFrame de séries temporais para o formato Prophet.
    Adiciona lags do IPCA como regressores para equiparar ao conjunto
    informacional do ARIMAX e LightGBM.
    
    Parâmetros
    ----------
    df             : DataFrame original (index = DatetimeIndex)
    target         : coluna alvo
    regressor_cols : lista de colunas já defasadas a incluir como regressores
    
    Retorna
    -------
    prophet_df : DataFrame no formato {ds, y, regressor_1, ..., regressor_n}
    """
    pdf = pd.DataFrame()
    pdf['ds'] = df.index
    pdf['y']  = df[target].values

    # Lags do IPCA como regressores (garantia de comparabilidade)
    for lag in [1, 2, 3]:
        pdf[f'ipca_lag_{lag}'] = df[target].shift(lag).values

    # Regressores externos defasados
    if regressor_cols:
        for col in regressor_cols:
            pdf[col] = df[col].values

    pdf = pdf.dropna().reset_index(drop=True)
    return pdf


# Especificação M1: só câmbio
reg_cols_m1 = ['usd_brl_lag4']

# Especificação M2: câmbio + CPI
reg_cols_m2 = ['usd_brl_lag4', 'us_cpi_lag2']

pdf_m1 = prepare_prophet_df(df_raw, regressor_cols=reg_cols_m1)
pdf_m2 = prepare_prophet_df(df_raw, regressor_cols=reg_cols_m2)

print(f'Prophet DF shape: {pdf_m2.shape}')
pdf_m2.head()

## 2. Seleção do modo de sazonalidade: aditivo vs. multiplicativo

Modo multiplicativo é esperado quando a amplitude das variações sazonais cresce proporcionalmente ao nível da série — padrão documentado em índices com energia e alimentos (Ha et al., 2020). Comparamos os dois modos via AIC aproximado sobre a janela de treino.

In [ ]:
def fit_prophet(pdf_train, regressors, seasonality_mode='multiplicative',
                changepoint_prior_scale=0.05, seasonality_prior_scale=10.0):
    """
    Instancia e treina um modelo Prophet com os parâmetros fornecidos.
    Inclui lags do IPCA e regressores externos.
    """
    m = Prophet(
        seasonality_mode=seasonality_mode,
        changepoint_prior_scale=changepoint_prior_scale,
        seasonality_prior_scale=seasonality_prior_scale,
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.95,
    )

    # Lags do IPCA como regressores
    for lag in [1, 2, 3]:
        m.add_regressor(f'ipca_lag_{lag}')

    # Regressores externos defasados
    for reg in regressors:
        m.add_regressor(reg)

    m.fit(pdf_train, iter=1000)
    return m


# Divisão treino para seleção de modo
train_end = '2019-12-31'
pdf_train_m2 = pdf_m2[pdf_m2['ds'] <= train_end]

print('Testando modo aditivo...')
m_add = fit_prophet(pdf_train_m2, reg_cols_m2, seasonality_mode='additive')

print('Testando modo multiplicativo...')
m_mul = fit_prophet(pdf_train_m2, reg_cols_m2, seasonality_mode='multiplicative')

# In-sample fit para comparação
for mode, m in [('aditivo', m_add), ('multiplicativo', m_mul)]:
    fc = m.predict(pdf_train_m2)
    mae_is = mean_absolute_error(pdf_train_m2['y'], fc['yhat'])
    print(f'  {mode}: MAE in-sample = {mae_is:.4f}')

print('\n→ Selecione o modo com menor MAE in-sample (confirmar via AIC se disponível).')

## 3. Otimização de hiperparâmetros (grid search)

Busca sobre `changepoint_prior_scale` e `seasonality_prior_scale` usando a função nativa `cross_validation` do Prophet.

In [ ]:
# Grade conforme Taylor & Letham (2017)
PARAM_GRID = {
    'changepoint_prior_scale':  [0.01, 0.1, 0.5],
    'seasonality_prior_scale':  [0.01, 0.1, 1.0],
    'seasonality_mode':         ['multiplicative'],  # definido na etapa anterior
}

grid_results = []

# Período inicial de treino para CV = janela de treino completa
# horizon = 3 meses (alinhado com H = 1, 2, 3 do estudo)
# period = 1 mês (rolling de 1 em 1)

for params in itertools.product(
    PARAM_GRID['changepoint_prior_scale'],
    PARAM_GRID['seasonality_prior_scale'],
    PARAM_GRID['seasonality_mode'],
):
    cps, sps, mode = params
    try:
        m = fit_prophet(
            pdf_train_m2, reg_cols_m2,
            seasonality_mode=mode,
            changepoint_prior_scale=cps,
            seasonality_prior_scale=sps,
        )

        # CV nativa do Prophet
        df_cv = cross_validation(
            m,
            initial=f'{len(pdf_train_m2) - 36} days',  # usa observações mensais
            period='30 days',
            horizon='90 days',  # ~3 meses
            disable_tqdm=True,
        )
        perf = performance_metrics(df_cv, rolling_window=1)
        cv_mae = perf['mae'].mean()

        grid_results.append({
            'changepoint_prior_scale': cps,
            'seasonality_prior_scale': sps,
            'seasonality_mode': mode,
            'cv_mae': cv_mae,
        })
        print(f'  cps={cps}, sps={sps} → CV MAE={cv_mae:.4f}')

    except Exception as e:
        print(f'  cps={cps}, sps={sps} → ERRO: {e}')

grid_df = pd.DataFrame(grid_results).sort_values('cv_mae')
print('\nTop 5 configurações:')
print(grid_df.head(5).to_string(index=False))

best_row = grid_df.iloc[0]
BEST_CPS  = best_row['changepoint_prior_scale']
BEST_SPS  = best_row['seasonality_prior_scale']
BEST_MODE = best_row['seasonality_mode']
print(f'\n→ Hiperparâmetros selecionados: cps={BEST_CPS}, sps={BEST_SPS}, mode={BEST_MODE}')

## 4. Expanding window — avaliação fora da amostra (h = 1, 2, 3)

Para cada t no período de teste, o Prophet é retrainado com a janela expansiva e gera previsões para h = 1, 2, 3. Previsões diretas: modelos separados por horizonte.

In [ ]:
def mase(actual, predicted, train_actual):
    naive_mae = np.mean(np.abs(np.diff(train_actual)))
    return np.mean(np.abs(actual - predicted)) / naive_mae


def expanding_window_prophet(pdf, reg_cols, horizon,
                              cps, sps, mode, train_end_date='2019-12-31'):
    """
    Expanding window para Prophet em horizonte h.
    Para cada t no período de teste:
      - Treina com dados até t - h
      - Gera forecast de h passos
      - Captura a previsão para t
    
    Retorna arrays de valores realizados e previstos.
    """
    test_rows = pdf[pdf['ds'] > pd.Timestamp(train_end_date)]
    preds, actuals = [], []

    for _, row in test_rows.iterrows():
        t = row['ds']
        cutoff = t - pd.DateOffset(months=horizon)

        train_w = pdf[pdf['ds'] <= cutoff]
        if len(train_w) < 36:
            continue

        try:
            m = fit_prophet(train_w, reg_cols, mode, cps, sps)

            # Future DataFrame: precisa dos regressores futuros
            # Usamos os valores já defasados disponíveis no período de previsão
            future = pdf[(pdf['ds'] > cutoff) &
                         (pdf['ds'] <= t)][['ds'] +
                         [f'ipca_lag_{l}' for l in [1, 2, 3]] +
                         reg_cols].copy()

            if len(future) < horizon:
                continue

            fc = m.predict(future)
            pred_val = fc.iloc[-1]['yhat']

        except Exception:
            pred_val = np.nan

        preds.append(pred_val)
        actuals.append(row['y'])

    return np.array(actuals), np.array(preds)


results_ew = {}
print('Iniciando expanding window Prophet (processo intensivo — pode demorar)...')

for h in [1, 2, 3]:
    # M1: só câmbio
    act, pred = expanding_window_prophet(
        pdf_m1, reg_cols_m1, h, BEST_CPS, BEST_SPS, BEST_MODE
    )
    results_ew[('M1_fx', h)] = {'actuals': act, 'preds': pred}
    print(f'  h={h} | M1_fx ✓ ({(~np.isnan(pred)).sum()} previsões)')

    # M2: câmbio + CPI
    act, pred = expanding_window_prophet(
        pdf_m2, reg_cols_m2, h, BEST_CPS, BEST_SPS, BEST_MODE
    )
    results_ew[('M2_fx_cpi', h)] = {'actuals': act, 'preds': pred}
    print(f'  h={h} | M2_fx_cpi ✓ ({(~np.isnan(pred)).sum()} previsões)')

print('Expanding window concluída.')

## 5. Métricas de avaliação

In [ ]:
train_ipca = pdf_m2[pdf_m2['ds'] <= '2019-12-31']['y'].values

rows = []
for (model_name, h), d in results_ew.items():
    act, pred = d['actuals'], d['preds']
    mask = ~np.isnan(pred)
    rows.append({
        'model':   model_name,
        'horizon': h,
        'MAE':     mean_absolute_error(act[mask], pred[mask]),
        'RMSE':    np.sqrt(mean_squared_error(act[mask], pred[mask])),
        'MASE':    mase(act[mask], pred[mask], train_ipca),
        'n':       mask.sum(),
    })

metrics_df = pd.DataFrame(rows).sort_values(['horizon', 'MAE'])
print('Métricas Prophet — expanding window:')
print(metrics_df.round(4).to_string(index=False))

## 6. Decomposição dos componentes do Prophet

A decomposição revela a contribuição de tendência, sazonalidade e regressores externos para a previsão — importante para a interpretação econômica solicitada pela orientadora.

In [ ]:
# Modelo final treinado sobre toda a janela de treino
print('Treinando modelo final para decomposição...')
m_final = fit_prophet(pdf_train_m2, reg_cols_m2, BEST_MODE, BEST_CPS, BEST_SPS)

# Previsão in-sample + decomposição
fc_train = m_final.predict(pdf_train_m2)

fig = m_final.plot_components(fc_train)
fig.suptitle('Prophet — Decomposição dos componentes (janela de treino)', fontsize=12)
plt.tight_layout()
plt.savefig('prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
Interpretação dos componentes:
- trend:        trajetória de longo prazo do IPCA; changepoints automáticos capturam rupturas.
- yearly:       padrão sazonal anual — leva em conta picos típicos de início de ano (reajustes).
- usd_brl_lag4: contribuição do câmbio defasado ao IPCA (pass-through cambial).
- us_cpi_lag2:  contribuição da inflação americana defasada (pressão externa).
""")

## 7. Análise por regime de volatilidade cambial (H2)

In [ ]:
df_raw['vol_cambio_12m'] = df_raw['usd_brl_pct'].rolling(12).std()
vol_threshold = df_raw[df_raw.index < '2020-01-01']['vol_cambio_12m'].median()
df_raw['regime_vol'] = np.where(df_raw['vol_cambio_12m'] > vol_threshold, 'alta', 'baixa')

print(f'Limiar de volatilidade (mediana treino): {vol_threshold:.4f}')

test_dates_raw = pdf_m2[pdf_m2['ds'] > '2019-12-31']['ds'].values

rows_regime = []
for (model_name, h), d in results_ew.items():
    act, pred = d['actuals'], d['preds']
    n = min(len(act), len(test_dates_raw))
    dates = pd.to_datetime(test_dates_raw[:n])
    regimes = df_raw.loc[dates[dates.isin(df_raw.index)], 'regime_vol'].values

    if len(regimes) != n:
        continue

    for regime in ['alta', 'baixa']:
        mask = (regimes == regime) & ~np.isnan(pred[:n])
        if mask.sum() < 3:
            continue
        rows_regime.append({
            'model':   model_name,
            'horizon': h,
            'regime':  regime,
            'MAE':     mean_absolute_error(act[:n][mask], pred[:n][mask]),
            'RMSE':    np.sqrt(mean_squared_error(act[:n][mask], pred[:n][mask])),
            'n':       mask.sum(),
        })

regime_df = pd.DataFrame(rows_regime).sort_values(['horizon', 'regime', 'MAE'])
print('\nMétricas por regime de volatilidade:')
print(regime_df.round(4).to_string(index=False))

## 8. Teste de Diebold-Mariano (H3 — ganho incremental do CPI)

In [ ]:
def diebold_mariano(e1, e2, h=1):
    """Teste DM (1995). H0: acurácia equivalente. DM < 0 → e2 menor."""
    d       = e1**2 - e2**2
    n       = len(d)
    d_mean  = np.mean(d)
    gamma0  = np.var(d, ddof=1)
    gammas  = [np.cov(d[k:], d[:-k])[0, 1] for k in range(1, h)]
    var_d   = max((gamma0 + 2 * sum(gammas)) / n, 1e-10)
    dm_stat = d_mean / np.sqrt(var_d)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return {'dm_stat': dm_stat, 'p_value': p_value}


print('Teste Diebold-Mariano — M1 (só câmbio) vs M2 (câmbio + CPI)')
print('DM < 0 → M2 melhor | p < 0.05 → rejeita H0')
print('-' * 65)

for h in [1, 2, 3]:
    m1_d = results_ew.get(('M1_fx', h))
    m2_d = results_ew.get(('M2_fx_cpi', h))
    if not m1_d or not m2_d:
        continue

    act   = m1_d['actuals']
    e1    = act - m1_d['preds']
    e2    = act - m2_d['preds']
    mask  = ~(np.isnan(e1) | np.isnan(e2))
    dm    = diebold_mariano(e1[mask], e2[mask], h=h)

    sig = '✓ SIGNIFICATIVO (5%)' if dm['p_value'] < 0.05 else '✗ não significativo'
    print(f'  h={h}: DM={dm["dm_stat"]:+.4f}, p={dm["p_value"]:.4f}  → {sig}')

## 9. Gráficos: observado vs. previsto

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
test_dates_plot = pd.to_datetime(test_dates_raw)

for i, h in enumerate([1, 2, 3]):
    ax   = axes[i]
    act  = results_ew[('M1_fx', h)]['actuals']
    p_m1 = results_ew[('M1_fx', h)]['preds']
    p_m2 = results_ew[('M2_fx_cpi', h)]['preds']

    n     = min(len(act), len(test_dates_plot))
    dates = test_dates_plot[:n]

    ax.plot(dates, act[:n],  color='black',      linewidth=1.2,  label='Realizado')
    ax.plot(dates, p_m1[:n], color='steelblue',  linewidth=0.9, linestyle='--', label='M1 — câmbio')
    ax.plot(dates, p_m2[:n], color='darkorange', linewidth=0.9, linestyle='--', label='M2 — câmbio+CPI')
    ax.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2021-12-01'),
               alpha=0.07, color='red', label='Choque COVID')
    ax.axhline(0, color='gray', linewidth=0.4)
    ax.set_title(f'Prophet — h={h} mês(es) à frente', fontsize=11)
    ax.set_ylabel('Var. % IPCA')
    ax.legend(fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.suptitle('Prophet — Previsões out-of-sample vs. realizadas (2020–2024)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('prophet_ew_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Exportação

In [ ]:
# Modelo final (treino completo)
with open('prophet_model_final.pkl', 'wb') as f:
    pickle.dump(m_final, f)

# Hiperparâmetros
best_hp = {
    'changepoint_prior_scale': BEST_CPS,
    'seasonality_prior_scale': BEST_SPS,
    'seasonality_mode':         BEST_MODE,
}
pd.DataFrame([best_hp]).to_csv('prophet_best_params.csv', index=False)

# Métricas
metrics_df.to_csv('prophet_metrics.csv', index=False)
regime_df.to_csv('prophet_metrics_regime.csv', index=False)
grid_df.to_csv('prophet_grid_search.csv', index=False)

# Previsões
for (model_name, h), d in results_ew.items():
    n = min(len(d['actuals']), len(test_dates_raw))
    pd.DataFrame({
        'date':      pd.to_datetime(test_dates_raw[:n]),
        'actual':    d['actuals'][:n],
        'predicted': d['preds'][:n],
    }).to_csv(f'prophet_preds_{model_name}_h{h}.csv', index=False)

print('Exportação concluída.')